##Cell 1 — Install + check GPU

In [ ]:
# Cell 1 — Install + check GPU + reproducibility setup

from google.colab import files
uploaded = files.upload()  # Upload raft_clean.jsonl (and optionally a pinned requirements file later)

# Install core packages (current run)
!pip -q install unsloth trl datasets accelerate
# Optional: avoids some audio dependency conflicts/warnings
!pip -q uninstall -y torchaudio

# -----------------------------
# Reproducibility: set seeds
# -----------------------------
import os, random
import numpy as np

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

random.seed(SEED)
np.random.seed(SEED)

# Torch is imported after install
import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Make some operations more deterministic (best effort)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("SEED:", SEED)

# -----------------------------
# Reproducibility: record versions
# -----------------------------
!python -V
!pip show torch transformers trl unsloth datasets accelerate | tee colab_versions.txt
!pip freeze | sort > colab_pip_freeze.txt

print("✅ Saved version logs: colab_versions.txt, colab_pip_freeze.txt")


##Cell 2 — Load + format dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="raft_clean.jsonl", split="train")
print("rows:", len(dataset))
print("cols:", dataset.column_names)
print("sample:", dataset[0])

# Use the SAME style as app.py / scripts/5_rag_cli.py
# Training prompt text now matches runtime prompt shape much better.
def format_example(ex):
    return {
        "text": (
            "You are a medical document QA assistant.\n"
            "RULES:\n"
            "- Use ONLY the SOURCES below.\n"
            "- If the answer is not clearly supported by the sources, say: "
            "\"I don't know based on the provided documents.\"\n"
            "- In your answer, cite sources like [S1], [S2] next to the claims they support.\n"
            "- Keep the answer concise and factual.\n\n"
            "QUESTION:\n"
            + ex["instruction"].strip() + "\n\n"
            "SOURCES:\n"
            + ex["input"].strip() + "\n\n"
            "ANSWER:\n"
            + ex["output"].strip()
        )
    }

dataset = dataset.map(format_example, remove_columns=dataset.column_names)

print(dataset[0]["text"][:1000])


##Cell 3 — Load model + LoRA

In [ ]:
from unsloth import FastLanguageModel

print("Using SEED =", SEED)

max_seq_length = 2048
dtype = None
load_in_4bit = True  # saves VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# If your installed unsloth version supports random_state, keep it.
# If it errors, remove random_state=SEED and rerun this cell.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0.0,
    random_state=SEED,   # <-- optional but helpful if supported
)

##Cell 4 — Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForLanguageModeling, set_seed

# Set Transformers seed too (important)
set_seed(SEED)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,   # effective batch = 16
        warmup_steps=10,
        max_steps=50,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir="outputs",
        save_steps=100,

        # Reproducibility / cleaner runs
        seed=SEED,
        data_seed=SEED,
        dataloader_num_workers=0,
        report_to="none",   # avoids wandb prompt
    ),
)

train_result = trainer.train()
print("✅ Training finished")
print(train_result)


##Cell 5 — Quick inference test (sanity check)

In [ ]:
# Cell 5 — Quick inference test (sanity check) using app/CLI-style prompt

FastLanguageModel.for_inference(model)

prompt = """You are a medical document QA assistant.
RULES:
- Use ONLY the SOURCES below.
- If the answer is not clearly supported by the sources, say: "I don't know based on the provided documents."
- In your answer, cite sources like [S1], [S2] next to the claims they support.
- Keep the answer concise and factual.

QUESTION:
What treatment is recommended in this example?

SOURCES:
[S1] pdf=example_guideline chunk_id=example_chunk_1
Hypertension guideline: start ACE inhibitor at X dose...

[S2] pdf=example_guideline chunk_id=example_chunk_2
Distractor text not relevant to the question.

ANSWER:
"""

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
out = model.generate(
    **inputs,
    max_new_tokens=120,
    do_sample=False,
)
print(tokenizer.decode(out[0], skip_special_tokens=True))


##Cell 6 — Save adapter (LoRA)

In [ ]:
model.save_pretrained("raft_lora_adapter")
tokenizer.save_pretrained("raft_lora_adapter")

# Add reproducibility metadata file
import json
from pathlib import Path
import platform

meta = {
    "base_model": "unsloth/Qwen2.5-0.5B-Instruct",
    "seed": SEED,
    "max_seq_length": max_seq_length,
    "load_in_4bit": load_in_4bit,
    "lora": {
        "r": 16,
        "alpha": 16,
        "dropout": 0.0,
        "target_modules": [
            "q_proj","k_proj","v_proj","o_proj",
            "gate_proj","up_proj","down_proj",
        ],
    },
    "training": {
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 8,
        "effective_batch_size": 16,
        "warmup_steps": 10,
        "max_steps": 300,
        "learning_rate": 2e-4,
        "fp16": True,
    },
    "dataset": {
        "rows": len(dataset),
        "text_field": "text",
        "source_file_uploaded": "raft_clean.jsonl",
    },
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
    },
}

Path("raft_lora_adapter/train_run_meta.json").write_text(
    json.dumps(meta, indent=2), encoding="utf-8"
)

print("✅ Saved metadata: raft_lora_adapter/train_run_meta.json")


##Cell 7 — OPTIONAL: merge + convert to GGUF + quantize

In [ ]:
# Merge LoRA into a full 16-bit model (optional, needed for GGUF conversion)
model.save_pretrained_merged("merged_model_16bit", tokenizer, save_method="merged_16bit")

!rm -rf /content/llama.cpp
!git clone https://github.com/ggerganov/llama.cpp

# Pin llama.cpp to the exact working commit (reproducible)
LLAMA_CPP_COMMIT = "244641955f6146f7e8474afff7772d427593a534"
!cd /content/llama.cpp && git checkout {LLAMA_CPP_COMMIT}

!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged_model_16bit \
  --outfile /content/qwen2.5-0.5b-raft-f16.gguf --outtype f16

!ls -lh /content/qwen2.5-0.5b-raft-f16.gguf

from google.colab import files
files.download("/content/qwen2.5-0.5b-raft-f16.gguf")

# Build quantizer + quantize to Q8_0
!rm -rf /content/llama.cpp/build
!cmake -S /content/llama.cpp -B /content/llama.cpp/build \
  -DCMAKE_BUILD_TYPE=Release \
  -DLLAMA_BUILD_TESTS=OFF \
  -DLLAMA_BUILD_EXAMPLES=OFF

!cmake --build /content/llama.cpp/build --target llama-quantize -j 1

!/content/llama.cpp/build/bin/llama-quantize \
  /content/qwen2.5-0.5b-raft-f16.gguf \
  /content/qwen2.5-0.5b-raft-q8_0.gguf \
  q8_0

!ls -lh /content/qwen2.5-0.5b-raft-q8_0.gguf
files.download("/content/qwen2.5-0.5b-raft-q8_0.gguf")


## Cell 8 — Export verification + metadata summary

In [ ]:
import os, json

build_info = {
    "base_model_family": "Qwen2.5-0.5B-Instruct (LoRA fine-tuned)",
    "llama_cpp_commit": "244641955f6146f7e8474afff7772d427593a534",
    "f16_gguf": "/content/qwen2.5-0.5b-raft-f16.gguf",
    "q8_0_gguf": "/content/qwen2.5-0.5b-raft-q8_0.gguf",
    "f16_exists": os.path.exists("/content/qwen2.5-0.5b-raft-f16.gguf"),
    "q8_0_exists": os.path.exists("/content/qwen2.5-0.5b-raft-q8_0.gguf"),
    "f16_size_mb": round(os.path.getsize("/content/qwen2.5-0.5b-raft-f16.gguf") / (1024**2), 2) if os.path.exists("/content/qwen2.5-0.5b-raft-f16.gguf") else None,
    "q8_0_size_mb": round(os.path.getsize("/content/qwen2.5-0.5b-raft-q8_0.gguf") / (1024**2), 2) if os.path.exists("/content/qwen2.5-0.5b-raft-q8_0.gguf") else None,
}

with open("/content/gguf_build_info.json", "w") as f:
    json.dump(build_info, f, indent=2)

print(json.dumps(build_info, indent=2))